In [3]:
pip install python-binance  

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\quang\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB 1.3 MB/s eta 0:00:08
   ---------------------------------------- 0.0/9.9 MB 960.0 kB/s eta 0:00:11
   ---------------------------------------- 0.0/9.9 MB 960.0 kB/s eta 0:00:11
    --------------------------------------- 0.1/9.9 MB 944.1 kB/s eta 0:00:11
    --------------------------------------- 0.2/9.9 MB 803.1 kB/s eta 0:00:13
    --------------------------------------- 0.2/9.9 MB 819.2 kB/s eta 0:00:12
   - -------------------------------------- 0.5/9.9 MB 1.6 MB/s eta 0:00:07
   -- ------------------------------------- 0.6/9.9 MB 1.8 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/9.9 MB 2.5 MB/s eta 0:00:04
   ---- ----------------------------------- 1.2/9.9 MB 2.7 MB/s eta 0:00:04
   -------- ------------------------------- 2.1/9.9 MB 4.3 MB/s eta 0:00:02
   ---------- ----------------------------- 2.5/9.9 MB 4.8 MB/s eta 0:00:02
   ------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\quang\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
from binance.client import Client
import time

# Khởi tạo Client (Không cần API Key cho dữ liệu công khai)
client = Client()

def download_binance_to_csv(symbol="BTCUSDT", interval="1m", duration="1 year ago UTC"):
    """
    symbol: Cặp tiền (BTCUSDT, ETHUSDT...)
    interval: Khung thời gian (1m, 5m, 1h, 1d)
    duration: Khoảng thời gian lùi về quá khứ
    """
    print(f"--- Bắt đầu tải dữ liệu {symbol} khung {interval} ---")
    print(f"--- Khoảng thời gian: {duration} ---")
    
    start_time = time.time()

    # 1. Gọi API lấy dữ liệu
    # Thư viện python-binance tự động xử lý việc chia nhỏ request (mỗi 1000 dòng) cho bạn
    klines = client.get_historical_klines(symbol, interval, duration)
    
    # 2. Chuyển thành DataFrame
    df = pd.DataFrame(klines, columns=[
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_asset_volume', 'number_of_trades',
        'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore'
    ])
    
    # 3. Làm sạch và định dạng
    # Chuyển timestamp sang định dạng datetime dễ đọc
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    
    # Ép kiểu dữ liệu số (Binance trả về string mặc định)
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)
    
    # Chỉ giữ lại các cột cần thiết cho lớp Silver/Gold
    final_df = df[['open_time', 'open', 'high', 'low', 'close', 'volume']]
    
    # 4. Xuất ra file CSV
    file_name = f"binance_{symbol}_{interval}_data.csv"
    final_df.to_csv(file_name, index=False)
    
    end_time = time.time()
    print(f"--- Hoàn thành! ---")
    print(f"Tổng số dòng: {len(final_df):,}")
    print(f"File đã lưu: {file_name}")
    print(f"Thời gian thực hiện: {round(end_time - start_time, 2)} giây")

# --- CHẠY THỬ ---
# Nếu muốn thử nhanh, bạn hãy để "1 day ago UTC" 
# Nếu muốn lấy đủ 1 năm cho đồ án, hãy để "1 year ago UTC"
download_binance_to_csv(symbol="BTCUSDT", interval=Client.KLINE_INTERVAL_1MINUTE, duration="1 day ago UTC")

--- Bắt đầu tải dữ liệu BTCUSDT khung 1m ---
--- Khoảng thời gian: 1 day ago UTC ---
--- Hoàn thành! ---
Tổng số dòng: 1,440
File đã lưu: binance_BTCUSDT_1m_data.csv
Thời gian thực hiện: 0.43 giây
